#  Naive Bayes Classification, Decision Tree, and Artificial Neural Network — Car Price Range

- Conrad Stanley Nowowiejski - 150891
-
Martyna Rutkowska - 122196

  - Language = Python

Methods and Implementation:
- Prodcued via DataBricks and exportred as HTML and iPython Notebook 
- All markdown cells present the approach used according to paratmers chosen
- Multiple layers and parts consist of the main goal of the project, as well as the added value with a third model (ANN)

## Goal: To determine which machine learning algorithm better classifies used cars into price categories, and to identify which vehicle attributes most strongly drive that classification.

Secondary Goal: Understanding different machine learning techniques, what drives these methods, and how models can improve results versus what the modesl are able to perform agains the provided features. 

  Layer 1 - Model Comparison: A Naive Bayes classifier and a Decision Tree were trained on used car auction data to classify cars as Budget, Mid-range, or Luxury, then evaluated against each other.

Layer 2 - Business Hypothesis: The comparative study pushed it further with a concrete business hypothesis — that usage-based factors (odometer reading and car age) and outperform subjective attributes (condition score, color, brand prestige) in predicting whether a car sells above the market average. This grounded the technical work in a real-world question relevant to dealerships and the used car industry. We pushed the boundaries by using an additional comparison model - the Artificial Neural Network, to strengthen our interpretations and set the benchmark.

Use of machine learning to classify used car auction prices, compare the performance of different algorithms, and identify which vehicle characteristics are the strongest predictors of value.

##  Part 1: Setup and Data

- `pandas` and `numpy` are the core data manipulation libraries — pandas handles the tabular dataset, numpy handles numerical operations.
- `matplotlib` and `seaborn` are used for all visualisations (bar charts, heatmaps, ROC curves, etc.).
- From `sklearn`, we import the three model classes: `GaussianNB` (Naive Bayes), `DecisionTreeClassifier`, and `MLPClassifier` (the neural network).
- `train_test_split` splits the data into training and test sets; `LabelEncoder` and `StandardScaler` handle preprocessing.
- The metrics imports (`accuracy_score`, `f1_score`, `roc_curve`, `auc`, etc.) give us the tools to evaluate and compare model performance.
- `permutation_importance` is specifically used later to measure feature importance for the ANN, since neural networks do not expose feature weights directly.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import (DecisionTreeClassifier,
                          plot_tree, export_text)
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, roc_curve, auc, f1_score
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelBinarizer

### 1. Load Data

- Reads the raw CSV file of used car auction records into a pandas DataFrame.
- The dataset contains **558,837 rows and 16 columns** — each row is one car sale at auction.
- Key columns include: `sellingprice` (the auction sale price), `mmr` (Manheim Market Report — the industry's wholesale value estimate), `odometer`, `condition`, `year`, `make`, `body`, `color`, and `transmission`.

In [0]:
df = pd.read_csv("car_prices.csv")
print(f"Raw dataset shape: {df.shape}")

### 2. Clean Data

- `df.dropna()` removes any rows with missing values — a necessary first step since all three models require complete records.
- The `remove_outliers` function uses the **IQR (Interquartile Range) method**: any value more than 1.5× the IQR below Q1 or above Q3 is considered an outlier and removed. This is applied to both `sellingprice` and `odometer`.
- The final filter `sellingprice > 100` removes near-zero prices that are likely data entry errors (e.g., internal transfers or bad records).
- After cleaning, **449,798 rows** remain — about 80% of the original dataset, indicating the data was already relatively clean.

In [0]:
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    return data[
        (data[column] >= Q1 - 1.5 * IQR) &
        (data[column] <= Q3 + 1.5 * IQR)
    ]

df_cleaned = df.dropna().copy()
df_cleaned = remove_outliers(df_cleaned, 'sellingprice')
df_cleaned = remove_outliers(df_cleaned, 'odometer')
df_cleaned = df_cleaned[df_cleaned['sellingprice'] > 100]
print(f"Cleaned dataset shape: {df_cleaned.shape}")

### 3. Target Variable 

- Converts the continuous `sellingprice` column into a three-class categorical target variable, which is required for classification.
- **Budget:** under $10,000 — the largest class by volume.
- **Mid:** $10,000–$29,999 — the most common price range in the dataset.
- **Luxury:** $30,000 and above — the smallest and most imbalanced class (only ~2,500 test cases vs ~54,000 Mid).
- This class imbalance (especially for Luxury) directly affects model performance — all three models struggle most with the Luxury class.

In [0]:
def assign_price_range(price):
    if price < 10000:   return "Budget"
    elif price < 30000: return "Mid"
    else:               return "Luxury"

df_cleaned['price_range'] = (
    df_cleaned['sellingprice'].apply(assign_price_range)
)

### 4. Feature Engineering

- Two new features are created that are more informative than the raw columns they derive from:
- `car_age` converts the model `year` into an age in years (subtracting from the maximum year in the dataset). Age is more directly meaningful than the year number for predicting depreciation.
- `mileage_per_year` normalises the odometer reading by age, capturing how heavily a car was used relative to how old it is. The `+ 1` avoids division-by-zero for brand-new cars (age = 0).
- Both engineered features ended up with low individual importance in the final models (dominated by `mmr`), but they represent sound domain reasoning about what drives car value.

In [0]:
df_cleaned['car_age'] = (
    df_cleaned['year'].max() - df_cleaned['year']
)
df_cleaned['mileage_per_year'] = (
    df_cleaned['odometer'] / (df_cleaned['car_age'] + 1)
)

### 5. Encode Categorical Features

- Machine learning models require numeric inputs, so categorical text columns must be converted to numbers.
- For each of the four categorical columns (`make`, `body`, `color`, `transmission`), only the **top 10 most frequent values** are kept; everything else is grouped into an `'Other'` category. This reduces noise from rare makes/colours that don't appear enough to be meaningful.
- `LabelEncoder` then converts each category to an integer (e.g., `Ford=3`, `Toyota=8`, `Other=0`). The new encoded columns are named `make_enc`, `body_enc`, etc.
- Important caveat: `LabelEncoder` assigns arbitrary integers, which implies an order that may not exist (e.g., it does not mean Ford > Toyota numerically). For tree-based models this is acceptable; for the ANN, StandardScaler later neutralises this issue.

In [0]:
le = LabelEncoder()
for col in ['make', 'body', 'color', 'transmission']:
    top_10 = df_cleaned[col].value_counts().index[:10]
    df_cleaned[col] = df_cleaned[col].where(
        df_cleaned[col].isin(top_10), 'Other'
    )
    df_cleaned[col + '_enc'] = le.fit_transform(
        df_cleaned[col].astype(str)
    )

### 6. Define Features and Target

- Defines the final set of **10 input features** used by all three models.
- `X` is the feature matrix (inputs); `y` is the target vector (the `price_range` labels).
- `mmr` is included because it is the industry's own wholesale price estimate — it captures market consensus on value and, as the feature importance analysis later confirms, it dominates all models.
- Note that `sellingprice` itself is excluded — it is what `price_range` is derived from, so including it would cause data leakage.

In [0]:
features = [
    'year', 'car_age', 'condition', 'odometer', 'mileage_per_year',
    'mmr', 'make_enc', 'body_enc', 'transmission_enc', 'color_enc'
]

X = df_cleaned[features]
y = df_cleaned['price_range']

### 7. Train/Test Split

- Splits the data into **80% training** (359,838 rows) and **20% test** (89,960 rows).
- `random_state=42` ensures the split is reproducible — running the notebook again produces identical splits.
- `stratify=y` ensures that the proportion of Budget, Mid, and Luxury cars is the same in both the training and test sets. This is important because Luxury is a small class — without stratification, it could end up under-represented in training.
- All three models are trained on `X_train`/`y_train` and evaluated on `X_test`/`y_test`, making the comparison fair.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✓ Data ready")
print(f"✓ X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"✓ Classes: {y.unique().tolist()}")

## Part 2: NBC vs Decision Tree

### 8.1. Class Distribution 

- Plots a bar chart showing how many cars fall into each price class.
- Mid-range cars are by far the most common class, followed by Budget; Luxury is a small minority.
- This class imbalance is important context for interpreting accuracy — a model that always predicted "Mid" would still achieve high raw accuracy, so we also track **Macro F1-score**, which weights all three classes equally.

In [0]:
plt.figure(figsize=(6, 4))
df_cleaned['price_range'].value_counts().plot(
    kind='bar', color=['#4C72B0', '#DD8452', '#55A868']
)
plt.title("Price Range Class Distribution")
plt.xlabel("Price Range")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()


### 8.2. Correlation Heatmap 

- Computes and visualises the pairwise Pearson correlation between the five most important continuous variables.
- Key finding: `mmr` and `sellingprice` are very strongly correlated (close to 1.0) — `mmr` is essentially a pre-computed market price, which is why it dominates the models.
- `odometer` is negatively correlated with price — higher mileage = lower value.
- `year` is positively correlated with price — newer cars are worth more.
- This heatmap also highlights why Naive Bayes is at a disadvantage: it assumes all features are independent of each other, but `mmr`, `year`, and `sellingprice` are clearly highly correlated.

In [0]:
plt.figure(figsize=(10, 8))
corr_cols = ['sellingprice', 'odometer', 'mmr', 'condition', 'year']
correlation_matrix = df_cleaned[corr_cols].corr()
sns.heatmap(
    correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f'
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

### 8.3. Depreciation Curve 

- Plots the average auction selling price at each car age (0 to ~14+ years).
- The curve is non-linear — depreciation is steepest in the first few years, then slows.
- The shaded confidence interval (added by `sns.lineplot` by default) shows that price variability is also highest for newer cars.

In [0]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_cleaned, x='car_age', y='sellingprice')
plt.title("Depreciation Curve: Average Price by Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Average Selling Price ($)")
plt.grid(True)
plt.tight_layout()
plt.savefig("depreciation_curve.png", dpi=150)
plt.show()

### 8.4. Depreciation Rate Per Year

- Quantifies the year-over-year depreciation rate as a percentage of the current year's average price.
- `.diff(-1)` computes the difference to the *next* row (i.e., how much the price falls moving from age N to age N+1).
- Key result: the average annual depreciation rate is **8.4%**, with the steepest single-year drop at **year 8 (–23.1%)**.
- Year 0–1 sees a large drop (~11.8%), confirming the well-known "new car" depreciation effect.

In [0]:
depreciation_df = df_cleaned.groupby(
    'car_age')['sellingprice'].mean().reset_index()
depreciation_df.columns = ['car_age', 'avg_price']
depreciation_df['price_drop'] = (
    depreciation_df['avg_price'].diff(-1)
)
depreciation_df['depreciation_rate_%'] = (
    depreciation_df['price_drop'] /
    depreciation_df['avg_price'] * 100
).round(2)

avg_annual_rate = depreciation_df['depreciation_rate_%'].mean()

print("=== Depreciation Analysis ===")
print(depreciation_df[
    ['car_age', 'avg_price', 'price_drop', 'depreciation_rate_%']
].head(15).to_string(index=False))
print(f"\nAverage Annual Rate: {avg_annual_rate:.2f}%")
print(
    f"Steepest Drop: Year "
    f"{depreciation_df['depreciation_rate_%'].idxmax()} "
    f"({depreciation_df['depreciation_rate_%'].max():.2f}%)"
)
print(
    f"Slowest Drop: Year "
    f"{depreciation_df['depreciation_rate_%'].idxmin()} "
    f"({depreciation_df['depreciation_rate_%'].min():.2f}%)"
)

plt.figure(figsize=(12, 5))
plt.bar(
    depreciation_df['car_age'].iloc[:-1],
    depreciation_df['depreciation_rate_%'].iloc[:-1],
    color='#DD8452', edgecolor='white'
)
plt.axhline(
    y=avg_annual_rate, color='red', linestyle='--',
    label=f'Average: {avg_annual_rate:.2f}%'
)
plt.title("Annual Depreciation Rate by Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Depreciation Rate (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("depreciation_rate.png", dpi=150)
plt.show()

### 9. Train NBC (no scaling)

- Trains a **Gaussian Naive Bayes** classifier — a probabilistic model that applies Bayes' theorem under the assumption that each feature is independent of the others and normally distributed.
- `var_smoothing=1e-9` adds a tiny amount to the variance of each feature to prevent division-by-zero in cases where a feature has near-zero variance. The default value is used here.
- No feature scaling is needed — NBC works with raw values.
- **Results:** Accuracy of **86.76%**, Macro F1 of **0.809**. The model performs well on Budget and Mid but struggles with Luxury precision (0.58) — it over-predicts Luxury, reflecting the independence assumption failing on correlated features like `mmr` and `year`.

In [0]:
gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train, y_train)
y_pred_nb = gnb.predict(X_test)

print("=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(classification_report(y_test, y_pred_nb))

### 10. Train Decision Tree (no sclaing)

- Trains a **Decision Tree** classifier using the Gini impurity criterion to determine the best split at each node.
- Key hyperparameter choices:
  - `max_depth=5` limits the tree to 5 levels — prevents overfitting by not allowing the tree to memorise the training data.
  - `min_samples_split=50` — a node must have at least 50 samples before it can be split.
  - `min_samples_leaf=20` — each leaf node must contain at least 20 samples, further preventing overfitting.
- No feature scaling is needed — decision trees are scale-invariant.
- **Results:** Accuracy of **93.96%**, Macro F1 of **0.889** — a large improvement over Naive Bayes.

In [0]:
dtree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42
)
dtree.fit(X_train, y_train)
y_pred_dt = dtree.predict(X_test)

print("=== Decision Tree Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))

### 11. NBC vs Decision Tree — Confusion Matrices 

- A **confusion matrix** shows, for each true class, how many predictions landed in each predicted class. The diagonal represents correct predictions; off-diagonal entries are errors.
- For Naive Bayes: the largest error source is Luxury — many Mid-range cars are misclassified as Luxury (high false positive rate for Luxury), explaining the low precision of 0.58.
- For the Decision Tree: the matrix is much more diagonal-dominant, indicating cleaner separations across all three classes. The Luxury row still has the most off-diagonal entries due to class imbalance.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_nb),
    display_labels=gnb.classes_
).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Naive Bayes — Confusion Matrix")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_dt),
    display_labels=dtree.classes_
).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Decision Tree — Confusion Matrix")

plt.tight_layout()
plt.savefig("nbc_dt_confusion_matrices.png", dpi=150)
plt.show()

### 12. NBC vs Decision Tree — Accuracy Comparison

- Produces a side-by-side bar chart comparing the two models' overall accuracy on the test set.
- The Decision Tree's bar (93.96%) clearly exceeds Naive Bayes (86.76%), providing a straightforward visual summary of model performance.
- Value labels are added above each bar using `plt.text()` for precise readability.

In [0]:
models_primary = ['Naive Bayes', 'Decision Tree']
accuracies_primary = [
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_dt)
]

plt.figure(figsize=(7, 4))
bars = plt.bar(
    models_primary, accuracies_primary,
    color=['#4C72B0', '#55A868'], width=0.4
)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("NBC vs Decision Tree — Accuracy Comparison")
for bar, acc in zip(bars, accuracies_primary):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{acc:.4f}", ha='center', fontsize=11
    )
plt.tight_layout()
plt.savefig("nbc_dt_accuracy.png", dpi=150)
plt.show()

### 13. Decision Tree — Tree Visualisation

- Renders the top 2 levels of the trained Decision Tree as a diagram, showing exactly how the model makes decisions.
- Each node shows: the splitting feature and threshold, the Gini impurity, the sample count, and the class distribution.
- `filled=True` colours each node by its majority class, making it easy to see where Budget, Mid, and Luxury predictions emerge.
- Only `max_depth=2` is shown here — displaying all 5 levels would be too dense to read. The full tree has up to 32 leaf nodes.

In [0]:
plt.figure(figsize=(22, 10))
plot_tree(
    dtree,
    feature_names=features,
    class_names=dtree.classes_,
    filled=True,
    fontsize=9,
    max_depth=2,
    impurity=True,
    proportion=False
)
plt.title("Decision Tree Logic — Top 2 Levels (Full depth=5)")
plt.tight_layout()
plt.savefig("decision_tree_plot.png", dpi=150)
plt.show()

### 14: Decision Tree — Text Rules (Decision Tree only, NBC is not possible)

- Exports the Decision Tree's logic as human-readable if/then rules — one of the key advantages of Decision Trees over black-box models.
- The output confirms that `mmr` is the very first (root) split and appears repeatedly throughout the top levels. The tree essentially asks: "How does this car's auction price compare to the industry's own wholesale estimate?"
- `condition` and `year` appear at deeper levels, fine-tuning classifications within MMR bands.
- This interpretability is the main reason Decision Trees are often preferred in business contexts — you can explain exactly why any prediction was made.

In [0]:
rules = export_text(dtree, feature_names=features, max_depth=3)
print("=== Decision Tree Rules (Top 3 Levels) ===")
print(rules)

### 15: NBC vs Decision Tree — Feature Importance

- `dtree.feature_importances_` returns the **Gini importance** for each feature — the total reduction in Gini impurity across all splits where that feature is used, normalised to sum to 1.
- **Result:** `mmr` accounts for **97.6%** of the tree's total importance. `condition` is a distant second at 2.2%. All other features — including `odometer`, `car_age`, `make_enc`, and `color_enc` — contribute essentially nothing.
- This is a striking finding: the Decision Tree has learned to rely almost entirely on the industry's pre-existing price estimate rather than on the raw vehicle attributes.

In [0]:
dt_importance = pd.DataFrame({
    'Feature':    features,
    'Importance': dtree.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance', y='Feature', data=dt_importance,
    hue='Feature', palette='viridis', legend=False
)
plt.title("Decision Tree — Gini Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("dt_feature_importance.png", dpi=150)
plt.show()

print("\nFeature Importance Ranking:")
print(dt_importance.to_string(index=False))

### 16. Decision Tree — Depth Sweep

- Trains 19 separate Decision Trees with `max_depth` ranging from 1 to 19, recording both training and test accuracy at each depth.
- The resulting plot shows the classic **bias-variance tradeoff**: at shallow depths, both training and test accuracy are low (underfitting); as depth increases, both improve rapidly; beyond a certain depth, training accuracy continues to climb while test accuracy plateaus or drops (overfitting).
- A vertical red line marks `depth=5` — our chosen model. The sweep justifies this choice: depth 5 sits at the "knee" of the test accuracy curve, capturing most of the available signal without overfitting.

In [0]:
train_scores, test_scores = [], []
depths = range(1, 20)

for d in depths:
    dt_temp = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_temp.fit(X_train, y_train)
    train_scores.append(
        accuracy_score(y_train, dt_temp.predict(X_train))
    )
    test_scores.append(
        accuracy_score(y_test, dt_temp.predict(X_test))
    )

plt.figure(figsize=(12, 5))
plt.plot(depths, train_scores,
         label='Training Accuracy', color='#DD8452', marker='o')
plt.plot(depths, test_scores,
         label='Test Accuracy', color='#4C72B0', marker='o')
plt.axvline(
    x=5, color='red', linestyle='--', label='Current max_depth=5'
)
plt.title("Decision Tree — Accuracy vs Tree Depth")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("depth_sweep.png", dpi=150)
plt.show()

### 17. NBC vs Decision Tree — ROC Curves

- **ROC (Receiver Operating Characteristic) curves** plot the True Positive Rate against the False Positive Rate at every possible classification threshold. The **AUC (Area Under the Curve)** summarises this: 1.0 is perfect, 0.5 is random.
- Because there are 3 classes, `LabelBinarizer` is used to convert `y_test` into a one-vs-rest binary format (one ROC curve per class).
- The Decision Tree shows higher AUC values across all three classes compared to Naive Bayes, confirming it is a better-calibrated model.
- Luxury typically has lower AUC than Budget/Mid due to the class imbalance.

In [0]:
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_roc_primary = [
    (gnb,   X_test, 'Naive Bayes',   '#4C72B0'),
    (dtree, X_test, 'Decision Tree', '#55A868'),
]

for ax, (model, x_data, label, color) in zip(
    axes, models_roc_primary
):
    y_score = model.predict_proba(x_data)
    for i, cls in enumerate(model.classes_):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_title(f'{label} — ROC Curves')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("nbc_dt_roc.png", dpi=150)
plt.show()

### 18: NBC vs Decision Tree — Primary Summary

- Compiles the key evaluation metrics for both models into a single summary table.
- **Accuracy** measures overall correctness; **Macro F1** averages the F1-score equally across all three classes (treating Luxury as equally important as Mid, despite its lower volume).
- Decision Tree wins on both metrics: Accuracy 93.96% vs 86.76%, Macro F1 0.889 vs 0.809.
- The gap is large enough to confidently conclude the Decision Tree is the better model for this task at this stage.

In [0]:
results_primary = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_nb,  average='macro'),
        f1_score(y_test, y_pred_dt,  average='macro')
    ]
})

print("=== NBC vs Decision Tree — Primary Comparison ===")
print(results_primary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(
    results_primary['Model'], results_primary['Accuracy'],
    color=['#4C72B0', '#55A868'], width=0.4
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Accuracy — NBC vs Decision Tree")
axes[0].set_ylabel("Accuracy")
for i, v in enumerate(results_primary['Accuracy']):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10)

axes[1].bar(
    results_primary['Model'], results_primary['Macro F1'],
    color=['#4C72B0', '#55A868'], width=0.4
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Macro F1 — NBC vs Decision Tree")
axes[1].set_ylabel("Macro F1 Score")
for i, v in enumerate(results_primary['Macro F1']):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10)

plt.tight_layout()
plt.savefig("nbc_dt_summary.png", dpi=150)
plt.show()

## Part 3: ANN — Standalone Deep Dive

The ANN is a more complex model included as a performance benchmark and to demonstrate what accuracy is achievable when interpretability is not a constraint.

### 19. Scale Features for ANN

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✓ Features scaled for ANN")
print(f"✓ Mean (first feature): {X_train_scaled[:, 0].mean():.4f}")
print(f"✓ Std  (first feature): {X_train_scaled[:, 0].std():.4f}")

- Neural networks are sensitive to the scale of input features — if `odometer` ranges from 0 to 200,000 and `condition` ranges from 1 to 5, the large-scale feature will dominate gradient updates during training.
- `StandardScaler` transforms each feature to have **mean = 0** and **standard deviation = 1** (z-score standardisation).
- Critically, the scaler is **fit only on the training data** (`fit_transform`) and then applied to the test data (`transform`). This prevents information about the test set from leaking into the scaling parameters.
- NBC and the Decision Tree do not require scaling (NBC uses parametric Gaussian distributions; Decision Trees are invariant to monotonic feature transformations).

### 20. Train ANN

- Trains a **Multi-Layer Perceptron (MLP)** — the most common type of feed-forward neural network.
- Architecture: input layer (10 features) → hidden layer 1 (128 neurons) → hidden layer 2 (64 neurons) → output layer (3 classes).
- `activation='relu'` — ReLU (Rectified Linear Unit) is the most common activation function for hidden layers; it is computationally efficient and avoids the vanishing gradient problem.
- `solver='adam'` — Adam is an adaptive gradient descent optimiser that adjusts the learning rate per parameter; it is the standard choice for MLPs.
- `alpha=0.001` — L2 regularisation penalty to prevent overfitting.
- `max_iter=300` allows up to 300 training epochs.
- **Results:** Accuracy of **94.65%**, Macro F1 of **0.897** — the highest of all three models, though only marginally better than the Decision Tree.

In [0]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

print("=== ANN (MLP) Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(classification_report(y_test, y_pred_mlp))


### 21. ANN — Confusion Matrix

- Displays the ANN's confusion matrix for visual inspection of where errors occur.
- Compared to both NBC and the Decision Tree, the ANN's matrix is the most diagonal-dominant — it makes fewer errors on Budget and Mid, and its Luxury F1 (0.79) matches the Decision Tree.
- The main remaining error pattern is some Mid cars misclassified as Budget (and vice versa), which is expected given the arbitrary price boundary at $10,000.

In [0]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_mlp),
    display_labels=mlp.classes_
).plot(colorbar=False)
plt.title("ANN (MLP) — Confusion Matrix")
plt.tight_layout()
plt.savefig("ann_confusion_matrix.png", dpi=150)
plt.show()

### 22. ANN — ROC Curves

- Plots ROC curves for the ANN across all three classes (one-vs-rest).
- The ANN produces well-calibrated probability estimates via `predict_proba`, which is required for ROC curve computation.
- AUC values for Budget and Mid are typically very high (>0.97), confirming the model is excellent at separating these classes from the rest. Luxury AUC is lower but still strong.

In [0]:
fig, ax = plt.subplots(figsize=(8, 6))
y_score_mlp = mlp.predict_proba(X_test_scaled)

for i, cls in enumerate(mlp.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score_mlp[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title("ANN (MLP) — ROC Curves Per Class")
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ann_roc.png", dpi=150)
plt.show()

### 23. ANN — Permutation Feature Importance

Uses binary target for importance analysis. Separate _b variables protect the original split.

- Neural networks do not have built-in feature importance scores like Decision Trees. **Permutation importance** is a model-agnostic method: for each feature, the values in the test set are randomly shuffled, and the drop in accuracy is measured. A larger drop = more important feature.
- The model is retrained on a **binary target** (above/below median price) for this analysis, using separate `_b` variables to avoid contaminating the main 3-class split.
- `n_repeats=10` shuffles each feature 10 times and averages the results for stability.
- **Key finding:** `mmr` is still the top feature (importance 0.432), but the ANN distributes importance more broadly than the Decision Tree — `condition` (0.028), `year` (0.012), `car_age` (0.006), and `odometer` (0.005) all register meaningful contributions. This suggests the ANN is learning subtler feature interactions.

In [0]:
median_price = df_cleaned['sellingprice'].median()
y_binary = (df_cleaned['sellingprice'] > median_price).astype(int)

X_b = df_cleaned[features]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_binary, test_size=0.2, random_state=42
)
X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled  = scaler.transform(X_test_b)

mlp_binary = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu',
    solver='adam', alpha=0.001, max_iter=300, random_state=42
)
mlp_binary.fit(X_train_b_scaled, y_train_b)

nn_perm = permutation_importance(
    mlp_binary, X_test_b_scaled, y_test_b,
    n_repeats=5, random_state=42
)
nn_importance = pd.DataFrame({
    'Feature':    features,
    'Importance': nn_perm.importances_mean
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance', y='Feature', data=nn_importance,
    hue='Feature', palette='plasma', legend=False
)
plt.title("ANN — Permutation Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ann_feature_importance.png", dpi=150)
plt.show()

print("\nANN — Top 5 Features:")
print(nn_importance.head(5).to_string(index=False))

### 24. ANN — Binary ROC vs NBC and DT

- Plots all three models' ROC curves on a single chart using a simplified **binary classification task** (above vs below median price), which allows a direct single-curve comparison.
- The ANN's curve sits highest (largest AUC), followed closely by the Decision Tree, with Naive Bayes lowest.
- This binary comparison is more visually intuitive than the three-class one-vs-rest setup and provides a clean summary of relative model quality.

In [0]:
dtree_b = DecisionTreeClassifier(max_depth=5, random_state=42)
dtree_b.fit(X_train_b, y_train_b)

gnb_b = GaussianNB(var_smoothing=1e-9)
gnb_b.fit(X_train_b, y_train_b)

plt.figure(figsize=(10, 6))
for model, x_data, label, color in [
    (mlp_binary, X_test_b_scaled, 'ANN (MLP)',    'purple'),
    (dtree_b,    X_test_b,        'Decision Tree', '#55A868'),
    (gnb_b,      X_test_b,        'Naive Bayes',   '#4C72B0'),
]:
    probs = model.predict_proba(x_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_b, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr, tpr,
        label=f'{label} (AUC = {roc_auc:.3f})', color=color
    )

plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC=0.500)')
plt.title('ROC Comparison — All Three Models (Binary Task)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("all_models_roc_binary.png", dpi=150)
plt.show()

## Part 4: Final Three-Model Summary

### 25. Three-Model Final Comparison

- Produces the definitive summary table of all three models across both evaluation metrics.
- **Final rankings:**
  - ANN: Accuracy 94.65%, Macro F1 0.897 — best overall performance
  - Decision Tree: Accuracy 93.96%, Macro F1 0.889 — near-equal to ANN, fully interpretable
  - Naive Bayes: Accuracy 86.76%, Macro F1 0.809 — reasonable baseline but clearly weaker
- The ANN's edge over the Decision Tree is small (~0.7% accuracy) — a meaningful trade-off given that the Decision Tree is fully interpretable and the ANN is a black box.
- The Decision Tree would be the preferred model in a business context where explainability matters; the ANN is appropriate when maximum accuracy is the priority.

In [0]:
results_final = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree', 'ANN (MLP)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_mlp)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_nb,  average='macro'),
        f1_score(y_test, y_pred_dt,  average='macro'),
        f1_score(y_test, y_pred_mlp, average='macro')
    ]
})

print("=== Final Three-Model Comparison ===")
print(results_final.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    results_final['Model'], results_final['Accuracy'],
    color=['#4C72B0', '#55A868', '#DD8452'], width=0.4
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Accuracy — All Three Models")
axes[0].set_ylabel("Accuracy")
for i, v in enumerate(results_final['Accuracy']):
    axes[0].text(
        i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10
    )

axes[1].bar(
    results_final['Model'], results_final['Macro F1'],
    color=['#4C72B0', '#55A868', '#DD8452'], width=0.4
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Macro F1 — All Three Models")
axes[1].set_ylabel("Macro F1 Score")
for i, v in enumerate(results_final['Macro F1']):
    axes[1].text(
        i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10
    )

plt.tight_layout()
plt.savefig("final_three_model_comparison.png", dpi=150)
plt.show()

### 26. Feature Importance — All Three Models

- Side-by-side bar charts compare feature importance rankings between the Decision Tree (Gini-based) and the ANN (permutation-based).
- Both models agree that `mmr` is the top predictor by a large margin. Both rank `condition` second.
- The key difference: the Decision Tree assigns `mmr` a near-total importance of **0.976**, leaving almost nothing for the other 9 features. The ANN distributes importance more broadly — `mmr` at 0.432, with `condition`, `year`, and `odometer` each contributing meaningfully.
- This suggests the ANN is capturing relationships between features that the shallow Decision Tree misses — which likely explains its marginally higher accuracy.


**Conclusion of business hypothesis:** `mmr` (a derived market estimate, not a raw vehicle attribute) overwhelmingly drives price classification. Among raw vehicle attributes, `condition` consistently ranks second, followed by `year`/`car_age`. Contrary to the initial hypothesis, `odometer` did not outperform `condition`.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    x='Importance', y='Feature', data=dt_importance,
    ax=axes[0], hue='Feature', palette='viridis', legend=False
)
axes[0].set_title('Decision Tree — Gini Importance')

sns.barplot(
    x='Importance', y='Feature', data=nn_importance,
    ax=axes[1], hue='Feature', palette='plasma', legend=False
)
axes[1].set_title('ANN — Permutation Importance')

plt.suptitle(
    "Feature Importance Comparison — Decision Tree vs ANN",
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig("importance_comparison.png", dpi=150)
plt.show()

print("\nDecision Tree vs ANN — Feature Agreement:")
merged = dt_importance.rename(
    columns={'Importance': 'DT_Importance'}
).merge(
    nn_importance.rename(columns={'Importance': 'ANN_Importance'}),
    on='Feature'
)
print(merged.to_string(index=False))